# sc4000 - inference (internet OFF)
Generated by `kaggle/build_notebooks.py` from commit `66670eb+uncommitted`. Do not edit by hand.

DINOv3 4-rotation pair scoring -> outlier removal -> rotation from pairs -> ALIKED + LightGlue @1024 -> COLMAP.
Set `IS_TRAIN = True` to run on train datasets and print the metric instead of writing the competition submission.

In [ ]:
import glob, os

def kaggle_input_dir(slug):
    for depth in range(4):
        pattern = os.path.join('/kaggle/input', *['*'] * depth, slug)
        matches = sorted(path for path in glob.glob(pattern) if os.path.isdir(path))
        if matches:
            return matches[0]
    raise FileNotFoundError(f'{slug} is not mounted under /kaggle/input; attached inputs are listed above')

!find /kaggle/input -maxdepth 3 -type d | head -40
COMPETITION_DIR = kaggle_input_dir('image-matching-challenge-2025')
LIGHTGLUE_PACKAGES_DIR = kaggle_input_dir('imc2024-packages-lightglue-rerun-kornia')
IMC25_UTILS_DIR = kaggle_input_dir('imc25-utils')
TRAINING_OUTPUT_DIR = kaggle_input_dir('sc4000-training')
print('COMPETITION_DIR =', COMPETITION_DIR)
print('LIGHTGLUE_PACKAGES_DIR =', LIGHTGLUE_PACKAGES_DIR)
print('IMC25_UTILS_DIR =', IMC25_UTILS_DIR)
print('TRAINING_OUTPUT_DIR =', TRAINING_OUTPUT_DIR)

In [ ]:
!pip install --no-index --no-deps {LIGHTGLUE_PACKAGES_DIR}/*
!pip install --no-index --no-deps {TRAINING_OUTPUT_DIR}/wheels/*.whl

In [ ]:
# lightglue loads weights through torch.hub; the LightGlue file name changed between releases, so provide both
!mkdir -p /root/.cache/torch/hub/checkpoints
!cp {TRAINING_OUTPUT_DIR}/aliked_lightglue_weights/aliked-n16.pth /root/.cache/torch/hub/checkpoints/
!cp {TRAINING_OUTPUT_DIR}/aliked_lightglue_weights/aliked_lightglue.pth /root/.cache/torch/hub/checkpoints/aliked_lightglue.pth
!cp {TRAINING_OUTPUT_DIR}/aliked_lightglue_weights/aliked_lightglue.pth /root/.cache/torch/hub/checkpoints/aliked_lightglue_v0-1_arxiv.pth

## Code (inlined from the repo by kaggle/build_notebooks.py)

In [ ]:
!mkdir -p /kaggle/working/sc4000/scripts /kaggle/working/sc4000/src/sc4000 /kaggle/working/sc4000/src/sc4000/models
!touch /kaggle/working/sc4000/src/sc4000/models/__init__.py

In [ ]:
%%writefile /kaggle/working/sc4000/scripts/run_pipeline.py
"""Run steps 1-6 over every dataset of a split and write submission.csv.

Models are loaded once and reused across datasets. A failing step skips the rest of that dataset
(its images become outliers) instead of killing the whole Kaggle run.

    python scripts/run_pipeline.py --split test
    python scripts/run_pipeline.py --split train --datasets ETs stairs imc2023_haiper
    python scripts/run_pipeline.py --split train --datasets ETs --start-step 4   # reuse steps 1-3 outputs
"""

import gc
import sys
import time
import traceback
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

import torch  # noqa: E402

from sc4000.config import build_argument_parser, config_from_arguments  # noqa: E402
from sc4000.dataset_io import dataset_image_dir, load_image_predictions_by_dataset  # noqa: E402
from sc4000.step1_pair_similarity_dinov3 import RotationAwarePairScorer, run_step1_pair_similarity  # noqa: E402
from sc4000.step2_outlier_removal_and_pair_selection import run_step2_outlier_removal_and_pair_selection  # noqa: E402
from sc4000.step3_absolute_rotation_from_pairs import run_step3_absolute_rotation  # noqa: E402
from sc4000.step4_local_matching_aliked_lightglue import AlikedLightGlueMatcher, run_step4_local_matching  # noqa: E402
from sc4000.step5_reconstruction_colmap import run_step5_reconstruction  # noqa: E402
from sc4000.step6_submission_and_scoring import run_step6_submission  # noqa: E402


def main() -> None:
    parser = build_argument_parser(__doc__.splitlines()[0], needs_dataset=False)
    parser.add_argument("--datasets", nargs="*", help="default: every dataset of the split that has an image folder")
    parser.add_argument("--start-step", type=int, default=1, choices=range(1, 7))
    arguments = parser.parse_args()
    config = config_from_arguments(arguments)
    split = arguments.split

    predictions_by_dataset = load_image_predictions_by_dataset(config, split)
    dataset_names = arguments.datasets or [
        name for name in predictions_by_dataset if dataset_image_dir(config, split, name).exists()
    ]
    print(f"[pipeline] split={split} datasets={dataset_names}")

    pair_scorer = RotationAwarePairScorer(config) if arguments.start_step <= 1 else None
    local_matcher = AlikedLightGlueMatcher(config) if arguments.start_step <= 4 else None

    for dataset_name in dataset_names:
        dataset_start_time = time.time()
        steps = [
            (1, lambda: run_step1_pair_similarity(config, split, dataset_name, pair_scorer)),
            (2, lambda: run_step2_outlier_removal_and_pair_selection(config, dataset_name)),
            (3, lambda: run_step3_absolute_rotation(config, dataset_name)),
            (4, lambda: run_step4_local_matching(config, split, dataset_name, local_matcher)),
            (5, lambda: run_step5_reconstruction(config, split, dataset_name)),
        ]
        for step_number, run_step in steps:
            if step_number < arguments.start_step:
                continue
            try:
                run_step()
            except Exception:
                print(f"[pipeline] {dataset_name}: step {step_number} failed, its images become outliers")
                traceback.print_exc()
                break
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"[pipeline] {dataset_name}: {time.time() - dataset_start_time:.1f}s")

    run_step6_submission(config, split, dataset_names)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/sc4000/scripts/train.py
"""Train the binary "same scene" PairClassifierHead on frozen DINOv3 descriptors.

1. Cache DINOv3 descriptors of every train image in its 4 rot90 versions
   (train_embeddings_dinov3_4rotations.h5). Datasets in --datasets-excluded-from-training
   (default ETs, stairs: they are also in data/test) are never loaded.
2. Each epoch samples pairs inside a dataset, balanced across datasets:
     50%  same scene (neither image an outlier), both rotated by the same random r   -> label 1
     25%  same scene, b rotated by r + d, d in {1, 2, 3}                             -> label 0
     25%  different scenes or an outlier image, both rotated by r                     -> label 0
   The rotated positives teach the head that only the right relative rotation matches, which
   is what step 1's argmax over 4 rotations relies on.
3. Validation on held-out train datasets: ROC-AUC of the step-1 max score, rotation accuracy
   on synthetically rotated positives, and outlier precision/recall of the step-2 rule.
4. Saves the best (mean val AUC) head to pair_classifier_head_dinov3.pt.

    python scripts/train.py --competition-data-dir /kaggle/input/image-matching-challenge-2025 \
        --dinov3-model-dir facebook/dinov3-vitb16-pretrain-lvd1689m --output-dir /kaggle/working/pair_classifier
"""

import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

import h5py  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
import torch.nn.functional as F  # noqa: E402
from tqdm import tqdm  # noqa: E402

from sc4000.config import ROTATION_COUNT, build_argument_parser, config_from_arguments  # noqa: E402
from sc4000.dataset_io import dataset_image_dir, labels_csv_path  # noqa: E402
from sc4000.models.dinov3_global_descriptor import Dinov3GlobalDescriptor  # noqa: E402
from sc4000.models.pair_classifier_head import PairClassifierHead, save_pair_classifier_head  # noqa: E402
from sc4000.step1_pair_similarity_dinov3 import score_all_pairs_with_four_rotations  # noqa: E402
from sc4000.step2_outlier_removal_and_pair_selection import find_outlier_image_mask  # noqa: E402

OUTLIER_SCENE_NAME = "outliers"


# ---------------------------------------------------------------------------------------- data


class TrainDataset:
    """One competition dataset: image names, scene labels and descriptors [N, 4, D]."""

    def __init__(self, name: str, image_names: "list[str]", scene_names: "list[str]", descriptors: np.ndarray):
        self.name = name
        self.image_names = image_names
        self.scene_names = np.array(scene_names)
        self.descriptors_in_four_rotations = descriptors
        upper_i, upper_j = np.triu_indices(len(image_names), k=1)
        is_outlier = self.scene_names == OUTLIER_SCENE_NAME
        same_scene = (self.scene_names[upper_i] == self.scene_names[upper_j]) & ~is_outlier[upper_i] & ~is_outlier[upper_j]
        self.positive_pairs = np.stack([upper_i[same_scene], upper_j[same_scene]], axis=1)
        self.negative_pairs = np.stack([upper_i[~same_scene], upper_j[~same_scene]], axis=1)

    def same_scene_label_matrix(self) -> np.ndarray:
        is_outlier = self.scene_names == OUTLIER_SCENE_NAME
        same_scene = self.scene_names[:, None] == self.scene_names[None, :]
        return same_scene & ~is_outlier[:, None] & ~is_outlier[None, :]


def cache_train_descriptors(config, cache_path: Path, dataset_names: "list[str]", labels: pd.DataFrame) -> None:
    """Compute missing DINOv3 descriptors into cache_path[dataset][image] -> [4, D]."""
    with h5py.File(cache_path, "a") as cache_file:
        missing = [
            (row.dataset, row.image) for row in labels.itertuples(index=False)
            if row.dataset in dataset_names and row.image not in cache_file.get(row.dataset, {})
        ]
        if not missing:
            return
        descriptor_model = Dinov3GlobalDescriptor(config.dinov3_model_dir, config.device, config.global_descriptor_resize_long_side)
        for dataset_name, image_name in tqdm(missing, desc="caching DINOv3 x4 rotations", dynamic_ncols=True):
            descriptors = descriptor_model.describe_image_in_four_rotations(
                dataset_image_dir(config, "train", dataset_name) / image_name
            )
            cache_file.require_group(dataset_name).create_dataset(image_name, data=descriptors)
        del descriptor_model
        torch.cuda.empty_cache()


def load_train_datasets(cache_path: Path, dataset_names: "list[str]", labels: pd.DataFrame) -> "list[TrainDataset]":
    train_datasets = []
    with h5py.File(cache_path, "r") as cache_file:
        for dataset_name in dataset_names:
            dataset_labels = labels[labels.dataset == dataset_name]
            descriptors = np.stack([cache_file[dataset_name][image][()] for image in dataset_labels.image])
            train_datasets.append(TrainDataset(dataset_name, list(dataset_labels.image), list(dataset_labels.scene), descriptors))
    return train_datasets


# ---------------------------------------------------------------------------------------- pair sampling


def sample_epoch_pairs(train_datasets: "list[TrainDataset]", pair_count: int, random_generator: np.random.Generator):
    """Returns global index arrays (a, b), rotations (rotation_a, rotation_b) and float labels."""
    dataset_offsets = np.cumsum([0] + [len(d.image_names) for d in train_datasets])
    kinds = random_generator.choice(3, size=pair_count, p=[0.5, 0.25, 0.25])  # 0 positive, 1 rotated positive, 2 negative
    index_a, index_b = np.zeros(pair_count, np.int64), np.zeros(pair_count, np.int64)
    valid = np.zeros(pair_count, bool)
    dataset_choice = random_generator.integers(len(train_datasets), size=pair_count)
    for dataset_index, train_dataset in enumerate(train_datasets):
        for kind in (0, 1, 2):
            selected = np.flatnonzero((dataset_choice == dataset_index) & (kinds == kind))
            pool = train_dataset.negative_pairs if kind == 2 else train_dataset.positive_pairs
            if len(selected) == 0 or len(pool) == 0:
                continue  # e.g. single-scene datasets have no negatives; those samples are dropped
            picked = pool[random_generator.integers(len(pool), size=len(selected))]
            swap = random_generator.random(len(selected)) < 0.5
            picked[swap] = picked[swap][:, ::-1]
            index_a[selected] = picked[:, 0] + dataset_offsets[dataset_index]
            index_b[selected] = picked[:, 1] + dataset_offsets[dataset_index]
            valid[selected] = True
    shared_rotation = random_generator.integers(ROTATION_COUNT, size=pair_count)
    rotation_offset = np.where(kinds == 1, random_generator.integers(1, ROTATION_COUNT, size=pair_count), 0)
    labels = (kinds == 0).astype(np.float32)
    return (
        index_a[valid], index_b[valid],
        shared_rotation[valid], ((shared_rotation + rotation_offset) % ROTATION_COUNT)[valid],
        labels[valid],
    )


# ---------------------------------------------------------------------------------------- validation


def roc_auc(scores: np.ndarray, labels: np.ndarray) -> float:
    positive_count, negative_count = labels.sum(), (~labels).sum()
    if positive_count == 0 or negative_count == 0:
        return float("nan")
    ranks = pd.Series(scores).rank().to_numpy()
    return float((ranks[labels].sum() - positive_count * (positive_count + 1) / 2) / (positive_count * negative_count))


@torch.no_grad()
def evaluate_head(head, val_datasets: "list[TrainDataset]", config, device) -> dict:
    head.eval()
    metrics = {}
    for val_dataset in val_datasets:
        pair_max_score, _ = score_all_pairs_with_four_rotations(val_dataset.descriptors_in_four_rotations, head, device)
        upper_i, upper_j = np.triu_indices(len(val_dataset.image_names), k=1)
        label_matrix = val_dataset.same_scene_label_matrix()
        auc = roc_auc(pair_max_score[upper_i, upper_j], label_matrix[upper_i, upper_j])

        # rotation accuracy: rotate image b of each positive pair by d; the argmax must undo it
        descriptors = torch.from_numpy(val_dataset.descriptors_in_four_rotations).float().to(device)
        positive_pairs = val_dataset.positive_pairs[:20000]
        correct, total = 0, 0
        for rotation_d in range(ROTATION_COUNT):
            upright_a = descriptors[positive_pairs[:, 0], 0][:, None].expand(-1, ROTATION_COUNT, -1)
            rotated_b = descriptors[positive_pairs[:, 1]][:, [(rotation_d + k) % ROTATION_COUNT for k in range(ROTATION_COUNT)]]
            predicted_k = head(upright_a, rotated_b).argmax(dim=1).cpu().numpy()
            correct += int((predicted_k == (ROTATION_COUNT - rotation_d) % ROTATION_COUNT).sum())
            total += len(positive_pairs)

        predicted_outlier = find_outlier_image_mask(pair_max_score, config.pair_similarity_threshold, config.min_neighbor_count_for_inlier)
        true_outlier = val_dataset.scene_names == OUTLIER_SCENE_NAME
        true_positive = int((predicted_outlier & true_outlier).sum())
        metrics[val_dataset.name] = {
            "auc": auc,
            "rotation_accuracy": correct / max(total, 1),
            "outlier_precision": true_positive / max(int(predicted_outlier.sum()), 1),
            "outlier_recall": true_positive / max(int(true_outlier.sum()), 1) if true_outlier.any() else float("nan"),
            "predicted_outlier_count": int(predicted_outlier.sum()),
            "true_outlier_count": int(true_outlier.sum()),
        }
    return metrics


# ---------------------------------------------------------------------------------------- main


def main() -> None:
    parser = build_argument_parser(__doc__.splitlines()[0], needs_dataset=False)
    parser.add_argument("--val-datasets", nargs="+", default=["imc2023_haiper", "imc2024_lizard_pond"])
    parser.add_argument("--output-dir", type=Path, default=Path("/kaggle/working/pair_classifier"))
    parser.add_argument("--epochs", type=int, default=30)
    parser.add_argument("--pairs-per-epoch", type=int, default=200_000)
    parser.add_argument("--batch-size", type=int, default=1024)
    parser.add_argument("--learning-rate", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--seed", type=int, default=0)
    arguments = parser.parse_args()
    config = config_from_arguments(arguments)
    arguments.output_dir.mkdir(parents=True, exist_ok=True)
    device = torch.device(config.device if torch.cuda.is_available() else "cpu")
    random_generator = np.random.default_rng(arguments.seed)
    torch.manual_seed(arguments.seed)

    labels = pd.read_csv(labels_csv_path(config, "train"))
    excluded = set(config.datasets_excluded_from_training)
    all_names = [name for name in labels.dataset.unique() if name not in excluded]
    val_names = [name for name in arguments.val_datasets if name in all_names]
    train_names = [name for name in all_names if name not in val_names]
    assert not excluded & (set(train_names) | set(val_names)), "excluded datasets leaked into training"
    print(f"train datasets: {train_names}\nval datasets:   {val_names}\nexcluded:       {sorted(excluded)}")

    cache_path = arguments.output_dir / "train_embeddings_dinov3_4rotations.h5"
    cache_train_descriptors(config, cache_path, train_names + val_names, labels)
    train_datasets = load_train_datasets(cache_path, train_names, labels)
    val_datasets = load_train_datasets(cache_path, val_names, labels)

    all_train_descriptors = torch.from_numpy(
        np.concatenate([d.descriptors_in_four_rotations for d in train_datasets])
    ).float().to(device)
    head = PairClassifierHead(input_dimension=all_train_descriptors.shape[-1]).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=arguments.learning_rate, weight_decay=arguments.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=arguments.epochs)

    best_mean_auc = -1.0
    checkpoint_path = arguments.output_dir / "pair_classifier_head_dinov3.pt"
    for epoch in range(1, arguments.epochs + 1):
        epoch_start_time = time.time()
        head.train()
        index_a, index_b, rotation_a, rotation_b, pair_labels = sample_epoch_pairs(
            train_datasets, arguments.pairs_per_epoch, random_generator
        )
        permutation = random_generator.permutation(len(pair_labels))
        epoch_loss = 0.0
        for batch_start in range(0, len(permutation), arguments.batch_size):
            batch = permutation[batch_start:batch_start + arguments.batch_size]
            descriptor_a = all_train_descriptors[index_a[batch], rotation_a[batch]]
            descriptor_b = all_train_descriptors[index_b[batch], rotation_b[batch]]
            target = torch.from_numpy(pair_labels[batch]).to(device)
            loss = F.binary_cross_entropy_with_logits(head(descriptor_a, descriptor_b), target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch)
        scheduler.step()

        val_metrics = evaluate_head(head, val_datasets, config, device)
        mean_auc = float(np.nanmean([m["auc"] for m in val_metrics.values()]))
        mean_rotation_accuracy = float(np.mean([m["rotation_accuracy"] for m in val_metrics.values()]))
        print(f"epoch {epoch:3d} loss {epoch_loss / len(permutation):.4f} val_auc {mean_auc:.4f} "
              f"val_rotation_acc {mean_rotation_accuracy:.4f} ({time.time() - epoch_start_time:.1f}s)")
        for name, metric_values in val_metrics.items():
            print(f"    {name}: " + " ".join(f"{key}={value:.3f}" if isinstance(value, float) else f"{key}={value}"
                                         for key, value in metric_values.items()))
        if mean_auc > best_mean_auc:
            best_mean_auc = mean_auc
            save_pair_classifier_head(head, checkpoint_path, {
                "backbone": "dinov3",
                "dinov3_model_dir": config.dinov3_model_dir,
                "global_descriptor_resize_long_side": config.global_descriptor_resize_long_side,
                "train_datasets": train_names,
                "val_datasets": val_names,
                "epoch": epoch,
                "val_metrics": val_metrics,
            })
            print(f"    saved best head -> {checkpoint_path}")
    print(f"best mean val AUC {best_mean_auc:.4f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/sc4000/src/sc4000/__init__.py
def main() -> None:
    print("Hello from sc4000!")


In [ ]:
%%writefile /kaggle/working/sc4000/src/sc4000/config.py
"""Single source of truth for paths and tunable knobs of the IMC2025 pipeline.

Every field of `PipelineConfig` is exposed as a `--field-name` command line flag
by `build_argument_parser`, so each step script can override any knob.
Kaggle runs Python 3.10, so keep the syntax 3.10-compatible.
"""

import argparse
import dataclasses
from dataclasses import dataclass, field
from pathlib import Path

ROTATION_COUNT = 4  # rotations by 0, 90, 180, 270 degrees counter-clockwise (np.rot90 / torch.rot90 convention)


@dataclass
class PipelineConfig:
    # ---------------------------------------------------------------- paths (Kaggle defaults)
    competition_data_dir: str = "/kaggle/input/image-matching-challenge-2025"
    working_dir: str = "/kaggle/working/result"
    submission_csv_path: str = "/kaggle/working/submission.csv"
    imc25_utils_dir: str = "/kaggle/input/imc25-utils"
    # outputs of the kaggle/training notebook, mounted in the inference notebook via kernel_sources
    dinov3_model_dir: str = "/kaggle/input/sc4000-training/dinov3-vitb16-pretrain-lvd1689m"
    pair_classifier_checkpoint_path: str = "/kaggle/input/sc4000-training/pair_classifier/pair_classifier_head_dinov3.pt"

    # ---------------------------------------------------------------- step 1: rotation-aware pair similarity (DINOv3)
    global_descriptor_resize_long_side: int = 448
    use_cosine_similarity_without_classifier: bool = False

    # ---------------------------------------------------------------- step 2: outlier removal and pair selection
    pair_similarity_threshold: float = 0.5
    min_neighbor_count_for_inlier: int = 2
    lightglue_pair_top_k: int = 10

    # ---------------------------------------------------------------- step 3: absolute rotation from pairwise rotations
    rotation_propagation_mode: str = "max_spanning_tree"  # "none" | "max_spanning_tree"

    # ---------------------------------------------------------------- step 4: ALIKED + LightGlue local matching
    local_feature_resize_long_side: int = 1024
    aliked_max_num_keypoints: int = 4096
    aliked_detection_threshold: float = 0.01
    lightglue_depth_confidence: float = -1.0
    lightglue_width_confidence: float = 0.99
    lightglue_min_match_score: float = 0.2
    min_matches_per_pair: int = 15

    # ---------------------------------------------------------------- step 5: COLMAP reconstruction
    colmap_camera_model: str = "simple-radial"  # simple-pinhole | pinhole | simple-radial (imc25-utils h5_to_db)
    colmap_min_model_size: int = 3
    colmap_max_num_models: int = 5

    # ---------------------------------------------------------------- training
    datasets_excluded_from_training: list = field(default_factory=lambda: ["ETs", "stairs"])

    # ---------------------------------------------------------------- runtime
    device: str = "cuda"

    def dataset_working_dir(self, dataset_name: str) -> Path:
        path = Path(self.working_dir) / dataset_name
        path.mkdir(parents=True, exist_ok=True)
        return path


# Intermediate file names, kept in one place so every step agrees on them.
STEP1_PAIR_SCORES_FILENAME = "step1_pair_scores_dinov3.npz"
STEP2_OUTLIER_IMAGES_FILENAME = "step2_outlier_images.json"
STEP2_SELECTED_PAIRS_FILENAME = "step2_selected_pairs.txt"
STEP3_ABSOLUTE_ROTATION_FILENAME = "step3_absolute_rotation_k.json"
STEP4_KEYPOINTS_FILENAME = "step4_keypoints_aliked_1024.h5"
STEP4_MATCHES_FILENAME = "step4_matches_aliked_lightglue_1024.h5"
STEP5_COLMAP_DIRNAME = "step5_colmap"
STEP5_REGISTERED_POSES_FILENAME = "step5_registered_poses.json"


def _parse_bool(value: str) -> bool:
    return value.lower() in ("1", "true", "yes")


def build_argument_parser(description: str, needs_dataset: bool = True) -> argparse.ArgumentParser:
    """Argument parser with --split, --dataset and one flag per PipelineConfig field."""
    parser = argparse.ArgumentParser(description=description)
    parser.add_argument("--split", choices=["train", "test"], default="test")
    if needs_dataset:
        parser.add_argument("--dataset", required=True, help="dataset name, e.g. ETs")
    default_config = PipelineConfig()
    for config_field in dataclasses.fields(PipelineConfig):
        flag = "--" + config_field.name.replace("_", "-")
        default_value = getattr(default_config, config_field.name)
        if isinstance(default_value, bool):
            parser.add_argument(flag, type=_parse_bool, default=default_value)
        elif isinstance(default_value, list):
            parser.add_argument(flag, nargs="*", default=default_value)
        else:
            parser.add_argument(flag, type=type(default_value), default=default_value)
    return parser


def config_from_arguments(arguments: argparse.Namespace) -> PipelineConfig:
    return PipelineConfig(**{
        config_field.name: getattr(arguments, config_field.name)
        for config_field in dataclasses.fields(PipelineConfig)
    })


In [ ]:
%%writefile /kaggle/working/sc4000/src/sc4000/dataset_io.py
"""Reading the competition CSVs, image paths and the rot90 keypoint geometry shared by all steps."""

import dataclasses
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sc4000.config import PipelineConfig


@dataclasses.dataclass
class ImagePrediction:
    image_id: "str | None"  # only used on the hidden test set
    dataset: str
    filename: str
    ground_truth_scene: "str | None" = None  # only known on the train split
    cluster_index: "int | None" = None
    rotation_matrix: "np.ndarray | None" = None
    translation_vector: "np.ndarray | None" = None


def labels_csv_path(config: PipelineConfig, split: str) -> Path:
    filename = "train_labels.csv" if split == "train" else "sample_submission.csv"
    return Path(config.competition_data_dir) / filename


def load_image_predictions_by_dataset(config: PipelineConfig, split: str) -> "dict[str, list[ImagePrediction]]":
    """{dataset: [ImagePrediction]} in CSV order. For the test split the scene column is meaningless."""
    labels = pd.read_csv(labels_csv_path(config, split))
    predictions_by_dataset: "dict[str, list[ImagePrediction]]" = {}
    for row in labels.itertuples(index=False):
        predictions_by_dataset.setdefault(row.dataset, []).append(ImagePrediction(
            image_id=None if split == "train" else row.image_id,
            dataset=row.dataset,
            filename=row.image,
            ground_truth_scene=row.scene if split == "train" else None,
        ))
    return predictions_by_dataset


def dataset_image_dir(config: PipelineConfig, split: str, dataset_name: str) -> Path:
    return Path(config.competition_data_dir) / split / dataset_name


def list_dataset_image_names(config: PipelineConfig, split: str, dataset_name: str) -> "list[str]":
    """Image file names of a dataset, taken from the CSV so train and test behave the same."""
    predictions_by_dataset = load_image_predictions_by_dataset(config, split)
    image_dir = dataset_image_dir(config, split, dataset_name)
    return [p.filename for p in predictions_by_dataset[dataset_name] if (image_dir / p.filename).exists()]


# ---------------------------------------------------------------------------------------- rot90 geometry


def rotated_image_shape(original_height: int, original_width: int, rotation_k: int) -> "tuple[int, int]":
    """(height, width) of np.rot90(image, rotation_k)."""
    return (original_width, original_height) if rotation_k % 2 == 1 else (original_height, original_width)


def map_keypoints_from_rotated_to_original(
    keypoints_xy_in_rotated_image: np.ndarray,
    original_height: int,
    original_width: int,
    rotation_k: int,
) -> np.ndarray:
    """Map (x, y) pixel-index coordinates in np.rot90(original, rotation_k) back into the original image.

    One counter-clockwise rot90 maps original pixel (x, y) of an image with width W to
    (y, W - 1 - x); its inverse is (x', y') -> (W - 1 - y', x'). Applying the inverse
    rotation_k times walks back through the intermediate rotated images.
    """
    keypoints_xy = np.asarray(keypoints_xy_in_rotated_image, dtype=np.float64).copy()
    rotation_k %= 4
    for step_index in range(rotation_k, 0, -1):
        # width of the image *before* this rotation step, i.e. rot90(original, step_index - 1)
        _, previous_width = rotated_image_shape(original_height, original_width, step_index - 1)
        x_rotated, y_rotated = keypoints_xy[:, 0].copy(), keypoints_xy[:, 1].copy()
        keypoints_xy[:, 0] = previous_width - 1 - y_rotated
        keypoints_xy[:, 1] = x_rotated
    return keypoints_xy


# ---------------------------------------------------------------------------------------- small json helpers


def write_json(path: Path, payload) -> None:
    path.write_text(json.dumps(payload, indent=1))


def read_json(path: Path):
    return json.loads(path.read_text())


In [ ]:
%%writefile /kaggle/working/sc4000/src/sc4000/models/dinov3_global_descriptor.py
"""Frozen DINOv3 global descriptor, computed for the 4 rot90 versions of every image."""

from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

from sc4000.config import ROTATION_COUNT

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class Dinov3GlobalDescriptor:
    """Descriptor = [L2(CLS token), L2(mean of patch tokens)] / sqrt(2), so it is unit length.

    The backbone is frozen. Images keep their aspect ratio: the long side is resized to
    `resize_long_side` and both sides are rounded to a multiple of the patch size.
    """

    def __init__(self, model_dir_or_hub_id: str, device: str, resize_long_side: int = 448):
        from transformers import AutoModel  # DINOv3 needs transformers >= 4.56

        self.device = torch.device(device if torch.cuda.is_available() or device == "cpu" else "cpu")
        self.model = AutoModel.from_pretrained(model_dir_or_hub_id).eval().to(self.device)
        for parameter in self.model.parameters():
            parameter.requires_grad_(False)
        self.patch_size = int(getattr(self.model.config, "patch_size", 16))
        self.num_register_tokens = int(getattr(self.model.config, "num_register_tokens", 0))
        self.resize_long_side = resize_long_side
        self.descriptor_dimension = 2 * int(self.model.config.hidden_size)

    def _load_normalized_image_tensor(self, image_path: Path) -> torch.Tensor:
        image = Image.open(image_path).convert("RGB")
        scale = self.resize_long_side / max(image.size)
        new_width = max(self.patch_size, round(image.width * scale / self.patch_size) * self.patch_size)
        new_height = max(self.patch_size, round(image.height * scale / self.patch_size) * self.patch_size)
        image = image.resize((new_width, new_height), Image.BICUBIC)
        tensor = torch.from_numpy(np.asarray(image, dtype=np.float32) / 255.0).permute(2, 0, 1)
        mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
        std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
        return (tensor - mean) / std

    @torch.no_grad()
    def _describe_batch(self, image_batch: torch.Tensor) -> torch.Tensor:
        with torch.autocast(device_type=self.device.type, dtype=torch.float16, enabled=self.device.type == "cuda"):
            tokens = self.model(pixel_values=image_batch.to(self.device)).last_hidden_state.float()
        cls_token = F.normalize(tokens[:, 0], dim=-1)
        patch_tokens = tokens[:, 1 + self.num_register_tokens:]
        mean_patch_token = F.normalize(patch_tokens.mean(dim=1), dim=-1)
        return torch.cat([cls_token, mean_patch_token], dim=-1) / np.sqrt(2.0)

    @torch.no_grad()
    def describe_image_in_four_rotations(self, image_path: Path) -> np.ndarray:
        """Returns float32 [ROTATION_COUNT, descriptor_dimension]; row k describes np.rot90(image, k)."""
        image_tensor = self._load_normalized_image_tensor(image_path)
        rotated = [torch.rot90(image_tensor, k, dims=(1, 2)) for k in range(ROTATION_COUNT)]
        # rotations 0/2 and 1/3 share a shape, so they can be batched together
        descriptors_even = self._describe_batch(torch.stack([rotated[0], rotated[2]]))
        descriptors_odd = self._describe_batch(torch.stack([rotated[1], rotated[3]]))
        descriptors = torch.stack([descriptors_even[0], descriptors_odd[0], descriptors_even[1], descriptors_odd[1]])
        return descriptors.cpu().numpy().astype(np.float32)


In [ ]:
%%writefile /kaggle/working/sc4000/src/sc4000/models/pair_classifier_head.py
"""Binary "same scene" classifier on top of frozen global descriptors."""

from pathlib import Path

import torch
import torch.nn as nn


class PairClassifierHead(nn.Module):
    """logit = MLP([pa, pb, pa * pb, |pa - pb|]) with pa, pb a shared projection of the two descriptors.

    The output is symmetrised, (f(a, b) + f(b, a)) / 2, so the score does not depend on pair order.
    """

    def __init__(self, input_dimension: int, projection_dimension: int = 256, hidden_dimension: int = 256, dropout: float = 0.1):
        super().__init__()
        self.head_config = {
            "input_dimension": input_dimension,
            "projection_dimension": projection_dimension,
            "hidden_dimension": hidden_dimension,
            "dropout": dropout,
        }
        self.projection = nn.Sequential(nn.Linear(input_dimension, projection_dimension), nn.GELU())
        self.classifier = nn.Sequential(
            nn.Linear(4 * projection_dimension, hidden_dimension),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dimension, 1),
        )

    def _ordered_logit(self, projected_a: torch.Tensor, projected_b: torch.Tensor) -> torch.Tensor:
        pair_features = torch.cat(
            [projected_a, projected_b, projected_a * projected_b, (projected_a - projected_b).abs()], dim=-1
        )
        return self.classifier(pair_features).squeeze(-1)

    def forward(self, descriptor_a: torch.Tensor, descriptor_b: torch.Tensor) -> torch.Tensor:
        """Returns logits; apply sigmoid for the same-scene probability."""
        projected_a = self.projection(descriptor_a)
        projected_b = self.projection(descriptor_b)
        return 0.5 * (self._ordered_logit(projected_a, projected_b) + self._ordered_logit(projected_b, projected_a))


def save_pair_classifier_head(head: PairClassifierHead, checkpoint_path: Path, extra_metadata: dict) -> None:
    torch.save({"state_dict": head.state_dict(), "head_config": head.head_config, **extra_metadata}, checkpoint_path)


def load_pair_classifier_head(checkpoint_path: Path, device: str) -> PairClassifierHead:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    head = PairClassifierHead(**checkpoint["head_config"])
    head.load_state_dict(checkpoint["state_dict"])
    return head.eval().to(device)


In [ ]:
%%writefile /kaggle/working/sc4000/src/sc4000/step1_pair_similarity_dinov3.py
"""STEP 1 - rotation-aware pair similarity with DINOv3.

Every pair (i, j) is compared 4 times: image i as-is against image j rotated by k * 90 degrees
counter-clockwise, k = 0..3. We keep the highest score and the k that produced it:

    pair_max_score[i, j]           = max_k  score(i, rot90(j, k))
    pair_relative_rotation_k[i, j] = argmax_k score(i, rot90(j, k))   ("rotate j by this k to look like i")

The matrices are mirrored, so pair_relative_rotation_k[j, i] = (4 - pair_relative_rotation_k[i, j]) % 4.

Output: <working_dir>/<dataset>/step1_pair_scores_dinov3.npz
"""

from pathlib import Path

import numpy as np
import torch
from tqdm import tqdm

from sc4000.config import (
    ROTATION_COUNT,
    STEP1_PAIR_SCORES_FILENAME,
    PipelineConfig,
    build_argument_parser,
    config_from_arguments,
)
from sc4000.dataset_io import dataset_image_dir, list_dataset_image_names
from sc4000.models.dinov3_global_descriptor import Dinov3GlobalDescriptor
from sc4000.models.pair_classifier_head import PairClassifierHead, load_pair_classifier_head


class RotationAwarePairScorer:
    """Bundles the frozen DINOv3 backbone and the (optional) trained pair classifier head."""

    def __init__(self, config: PipelineConfig):
        self.descriptor_model = Dinov3GlobalDescriptor(
            config.dinov3_model_dir, config.device, config.global_descriptor_resize_long_side
        )
        self.device = self.descriptor_model.device
        self.pair_classifier_head: "PairClassifierHead | None" = None
        checkpoint_path = Path(config.pair_classifier_checkpoint_path)
        if config.use_cosine_similarity_without_classifier:
            print("[step1] using cosine similarity (classifier disabled by config)")
        elif checkpoint_path.exists():
            self.pair_classifier_head = load_pair_classifier_head(checkpoint_path, str(self.device))
            print(f"[step1] loaded pair classifier head from {checkpoint_path}")
        else:
            print(f"[step1] WARNING: no classifier at {checkpoint_path}, falling back to cosine similarity. "
                  "Tune --pair-similarity-threshold for cosine scores.")

    def describe_images(self, image_paths: "list[Path]") -> np.ndarray:
        """float32 [N, ROTATION_COUNT, D]."""
        descriptors = [
            self.descriptor_model.describe_image_in_four_rotations(path)
            for path in tqdm(image_paths, desc="[step1] DINOv3 x4 rotations", dynamic_ncols=True)
        ]
        return np.stack(descriptors)


@torch.no_grad()
def score_all_pairs_with_four_rotations(
    descriptors_in_four_rotations: np.ndarray,
    pair_classifier_head: "PairClassifierHead | None",
    device: "torch.device | str" = "cpu",
) -> "tuple[np.ndarray, np.ndarray]":
    """descriptors [N, 4, D] -> (pair_max_score [N, N], pair_relative_rotation_k [N, N]); diagonal = -inf / 0."""
    descriptors = torch.from_numpy(descriptors_in_four_rotations).float().to(device)
    image_count, rotation_count, descriptor_dimension = descriptors.shape
    all_rotated_descriptors = descriptors.reshape(image_count * rotation_count, descriptor_dimension)

    # score_per_rotation[i, j, k] = score(i at rotation 0, j at rotation k)
    score_per_rotation = torch.empty(image_count, image_count, rotation_count, device=device)
    for i in range(image_count):
        upright_descriptor_i = descriptors[i, 0].expand_as(all_rotated_descriptors)
        if pair_classifier_head is None:
            row_scores = (upright_descriptor_i * all_rotated_descriptors).sum(dim=-1)
        else:
            row_scores = torch.sigmoid(pair_classifier_head(upright_descriptor_i, all_rotated_descriptors))
        score_per_rotation[i] = row_scores.view(image_count, rotation_count)

    pair_max_score_full, pair_relative_rotation_k_full = score_per_rotation.max(dim=-1)
    pair_max_score_full = pair_max_score_full.cpu().numpy()
    pair_relative_rotation_k_full = pair_relative_rotation_k_full.cpu().numpy()

    # Keep the upper triangle and mirror it, so (i, j) and (j, i) always agree.
    pair_max_score = np.full((image_count, image_count), -np.inf, dtype=np.float32)
    pair_relative_rotation_k = np.zeros((image_count, image_count), dtype=np.int64)
    upper_i, upper_j = np.triu_indices(image_count, k=1)
    pair_max_score[upper_i, upper_j] = pair_max_score_full[upper_i, upper_j]
    pair_max_score[upper_j, upper_i] = pair_max_score_full[upper_i, upper_j]
    pair_relative_rotation_k[upper_i, upper_j] = pair_relative_rotation_k_full[upper_i, upper_j]
    pair_relative_rotation_k[upper_j, upper_i] = (ROTATION_COUNT - pair_relative_rotation_k_full[upper_i, upper_j]) % ROTATION_COUNT
    return pair_max_score, pair_relative_rotation_k


def run_step1_pair_similarity(
    config: PipelineConfig, split: str, dataset_name: str, pair_scorer: "RotationAwarePairScorer | None" = None
) -> Path:
    pair_scorer = pair_scorer or RotationAwarePairScorer(config)
    image_names = list_dataset_image_names(config, split, dataset_name)
    image_dir = dataset_image_dir(config, split, dataset_name)
    descriptors_in_four_rotations = pair_scorer.describe_images([image_dir / name for name in image_names])
    pair_max_score, pair_relative_rotation_k = score_all_pairs_with_four_rotations(
        descriptors_in_four_rotations, pair_scorer.pair_classifier_head, pair_scorer.device
    )
    output_path = config.dataset_working_dir(dataset_name) / STEP1_PAIR_SCORES_FILENAME
    np.savez(
        output_path,
        image_names=np.array(image_names),
        pair_max_score=pair_max_score,
        pair_relative_rotation_k=pair_relative_rotation_k,
    )
    print(f"[step1] {dataset_name}: scored {len(image_names)} images -> {output_path}")
    return output_path


def load_step1_pair_scores(config: PipelineConfig, dataset_name: str) -> "tuple[list[str], np.ndarray, np.ndarray]":
    with np.load(config.dataset_working_dir(dataset_name) / STEP1_PAIR_SCORES_FILENAME) as step1_output:
        return (
            [str(name) for name in step1_output["image_names"]],
            step1_output["pair_max_score"],
            step1_output["pair_relative_rotation_k"],
        )


def main() -> None:
    arguments = build_argument_parser(__doc__.splitlines()[1]).parse_args()
    run_step1_pair_similarity(config_from_arguments(arguments), arguments.split, arguments.dataset)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/sc4000/src/sc4000/step2_outlier_removal_and_pair_selection.py
"""STEP 2 - outlier removal and pair selection for local matching.

An image is an outlier when fewer than `min_neighbor_count_for_inlier` other images score above
`pair_similarity_threshold` with it. Outliers are reported as "outliers" and skipped by later steps.

Pairs kept for LightGlue (inlier images only):
  * every pair with pair_max_score > pair_similarity_threshold, plus
  * each inlier's `lightglue_pair_top_k` best-scoring inlier neighbours, so no inlier is left isolated.

Outputs: step2_outlier_images.json (list of names), step2_selected_pairs.txt ("name_a name_b" per line)
"""

import numpy as np

from sc4000.config import (
    STEP2_OUTLIER_IMAGES_FILENAME,
    STEP2_SELECTED_PAIRS_FILENAME,
    PipelineConfig,
    build_argument_parser,
    config_from_arguments,
)
from sc4000.dataset_io import read_json, write_json
from sc4000.step1_pair_similarity_dinov3 import load_step1_pair_scores


def find_outlier_image_mask(
    pair_max_score: np.ndarray, pair_similarity_threshold: float, min_neighbor_count_for_inlier: int
) -> np.ndarray:
    """bool [N]; True = outlier."""
    neighbor_count_above_threshold = (pair_max_score > pair_similarity_threshold).sum(axis=1)
    return neighbor_count_above_threshold < min_neighbor_count_for_inlier


def select_pairs_for_local_matching(
    pair_max_score: np.ndarray, outlier_image_mask: np.ndarray, pair_similarity_threshold: float, lightglue_pair_top_k: int
) -> "list[tuple[int, int]]":
    """Sorted list of (i, j) with i < j, both inliers."""
    inlier_indices = np.flatnonzero(~outlier_image_mask)
    inlier_scores = pair_max_score[np.ix_(inlier_indices, inlier_indices)]
    selected_pairs: "set[tuple[int, int]]" = set()

    above_i, above_j = np.nonzero(inlier_scores > pair_similarity_threshold)
    for local_i, local_j in zip(above_i, above_j):
        if local_i < local_j:
            selected_pairs.add((int(inlier_indices[local_i]), int(inlier_indices[local_j])))

    neighbor_count = min(lightglue_pair_top_k, len(inlier_indices) - 1)
    if neighbor_count > 0:
        top_neighbors = np.argsort(-inlier_scores, axis=1)[:, :neighbor_count]  # diagonal is -inf, never picked first
        for local_i, neighbors in enumerate(top_neighbors):
            for local_j in neighbors:
                if local_i == local_j:
                    continue
                image_a, image_b = int(inlier_indices[local_i]), int(inlier_indices[local_j])
                selected_pairs.add((min(image_a, image_b), max(image_a, image_b)))
    return sorted(selected_pairs)


def run_step2_outlier_removal_and_pair_selection(config: PipelineConfig, dataset_name: str) -> None:
    image_names, pair_max_score, _ = load_step1_pair_scores(config, dataset_name)
    outlier_image_mask = find_outlier_image_mask(
        pair_max_score, config.pair_similarity_threshold, config.min_neighbor_count_for_inlier
    )
    selected_pairs = select_pairs_for_local_matching(
        pair_max_score, outlier_image_mask, config.pair_similarity_threshold, config.lightglue_pair_top_k
    )
    output_dir = config.dataset_working_dir(dataset_name)
    write_json(output_dir / STEP2_OUTLIER_IMAGES_FILENAME, [image_names[i] for i in np.flatnonzero(outlier_image_mask)])
    (output_dir / STEP2_SELECTED_PAIRS_FILENAME).write_text(
        "".join(f"{image_names[i]} {image_names[j]}\n" for i, j in selected_pairs)
    )
    inlier_count = int((~outlier_image_mask).sum())
    print(f"[step2] {dataset_name}: {inlier_count}/{len(image_names)} inliers, "
          f"{len(selected_pairs)} pairs selected for LightGlue")
    if inlier_count == 0:
        print(f"[step2] WARNING: every image of {dataset_name} is an outlier "
              f"(min_neighbor_count_for_inlier={config.min_neighbor_count_for_inlier}, {len(image_names)} images)")


def load_step2_outputs(config: PipelineConfig, dataset_name: str) -> "tuple[list[str], list[tuple[str, str]]]":
    output_dir = config.dataset_working_dir(dataset_name)
    outlier_image_names = read_json(output_dir / STEP2_OUTLIER_IMAGES_FILENAME)
    selected_pairs = [
        tuple(line.split()) for line in (output_dir / STEP2_SELECTED_PAIRS_FILENAME).read_text().splitlines() if line.strip()
    ]
    return outlier_image_names, selected_pairs


def main() -> None:
    arguments = build_argument_parser(__doc__.splitlines()[1]).parse_args()
    run_step2_outlier_removal_and_pair_selection(config_from_arguments(arguments), arguments.dataset)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/sc4000/src/sc4000/step3_absolute_rotation_from_pairs.py
"""STEP 3 - absolute rot90 per image from the pairwise relative rotations of step 1.

absolute_rotation_k[x] is the number of counter-clockwise 90 degree turns that brings image x into
the shared frame of its component. If rot90(j, r) looks like i (r = pair_relative_rotation_k[i, j]),
then absolute_rotation_k[j] = (absolute_rotation_k[i] + r) % 4.

Mode "max_spanning_tree": per connected component of the selected-pair graph (edge weight = pair
score), propagate along the maximum spanning tree from its highest-degree node, then add the global
offset c that leaves the most images unrotated (most photos are upright).
Mode "none": every image gets k = 0.

Output: step3_absolute_rotation_k.json  {image_name: k}  (inlier images only)
"""

from collections import Counter

import networkx as nx
import numpy as np

from sc4000.config import (
    ROTATION_COUNT,
    STEP3_ABSOLUTE_ROTATION_FILENAME,
    PipelineConfig,
    build_argument_parser,
    config_from_arguments,
)
from sc4000.dataset_io import read_json, write_json
from sc4000.step1_pair_similarity_dinov3 import load_step1_pair_scores
from sc4000.step2_outlier_removal_and_pair_selection import load_step2_outputs


def propagate_absolute_rotation_on_max_spanning_tree(
    image_count: int,
    selected_pair_indices: "list[tuple[int, int]]",
    pair_max_score: np.ndarray,
    pair_relative_rotation_k: np.ndarray,
) -> "dict[int, int]":
    """{image_index: absolute_rotation_k} for every image that appears in a selected pair."""
    pair_graph = nx.Graph()
    for image_a, image_b in selected_pair_indices:
        pair_graph.add_edge(image_a, image_b, weight=float(pair_max_score[image_a, image_b]))

    absolute_rotation_k: "dict[int, int]" = {}
    for component_nodes in nx.connected_components(pair_graph):
        component_tree = nx.maximum_spanning_tree(pair_graph.subgraph(component_nodes), weight="weight")
        root = max(component_tree.degree, key=lambda node_and_degree: node_and_degree[1])[0]
        component_rotation_k = {root: 0}
        for parent, child in nx.bfs_edges(component_tree, root):
            component_rotation_k[child] = (component_rotation_k[parent] + int(pair_relative_rotation_k[parent, child])) % ROTATION_COUNT

        # choose the offset that leaves the largest number of images unrotated; ties keep offset 0
        rotation_histogram = Counter(component_rotation_k.values())
        best_offset = max(range(ROTATION_COUNT), key=lambda offset: (rotation_histogram[(-offset) % ROTATION_COUNT], -offset))
        for image_index, rotation_k in component_rotation_k.items():
            absolute_rotation_k[image_index] = (rotation_k + best_offset) % ROTATION_COUNT
    return absolute_rotation_k


def run_step3_absolute_rotation(config: PipelineConfig, dataset_name: str) -> None:
    image_names, pair_max_score, pair_relative_rotation_k = load_step1_pair_scores(config, dataset_name)
    outlier_image_names, selected_pairs = load_step2_outputs(config, dataset_name)
    name_to_index = {name: index for index, name in enumerate(image_names)}
    outlier_set = set(outlier_image_names)
    inlier_image_names = [name for name in image_names if name not in outlier_set]

    absolute_rotation_k_by_name = {name: 0 for name in inlier_image_names}
    if config.rotation_propagation_mode == "max_spanning_tree":
        selected_pair_indices = [(name_to_index[a], name_to_index[b]) for a, b in selected_pairs]
        propagated = propagate_absolute_rotation_on_max_spanning_tree(
            len(image_names), selected_pair_indices, pair_max_score, pair_relative_rotation_k
        )
        for image_index, rotation_k in propagated.items():
            absolute_rotation_k_by_name[image_names[image_index]] = rotation_k
    elif config.rotation_propagation_mode != "none":
        raise ValueError(f"unknown rotation_propagation_mode {config.rotation_propagation_mode!r}")

    write_json(config.dataset_working_dir(dataset_name) / STEP3_ABSOLUTE_ROTATION_FILENAME, absolute_rotation_k_by_name)
    rotated_image_names = {name: k for name, k in absolute_rotation_k_by_name.items() if k != 0}
    print(f"[step3] {dataset_name}: {len(rotated_image_names)}/{len(absolute_rotation_k_by_name)} images rotated "
          f"({config.rotation_propagation_mode}) {rotated_image_names if len(rotated_image_names) <= 10 else ''}")


def load_step3_absolute_rotation(config: PipelineConfig, dataset_name: str) -> "dict[str, int]":
    return read_json(config.dataset_working_dir(dataset_name) / STEP3_ABSOLUTE_ROTATION_FILENAME)


def main() -> None:
    arguments = build_argument_parser(__doc__.splitlines()[1]).parse_args()
    run_step3_absolute_rotation(config_from_arguments(arguments), arguments.dataset)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/sc4000/src/sc4000/step4_local_matching_aliked_lightglue.py
"""STEP 4 - ALIKED + LightGlue local matching at a single resize (long side 1024).

Each inlier image is turned upright with np.rot90(image, absolute_rotation_k) from step 3, then
ALIKED runs on the rotated image. LightGlue matches the step-2 pairs in that rotated frame,
because its positional encoding is not rotation invariant. Keypoints are stored **mapped back to
original-image pixels**, so COLMAP runs on the untouched images and its poses are directly in the
ground-truth orientation.

Outputs (IMC baseline layout, keys are image file names):
  step4_keypoints_aliked_1024.h5            keypoints[name] -> float32 [N, 2] (x, y) in the original image
  step4_matches_aliked_lightglue_1024.h5    matches[name_a][name_b] -> int64 [M, 2]

Self test: `--selftest` checks the rot90 keypoint back-mapping on a random array, and
`--selftest-image PATH` also matches PATH against rotated copies of itself, turned upright the way
step 3 would, and checks the back-mapped keypoints against the known rotation.
"""

from pathlib import Path

import h5py
import numpy as np
import torch
from tqdm import tqdm

from sc4000.config import (
    ROTATION_COUNT,
    STEP4_KEYPOINTS_FILENAME,
    STEP4_MATCHES_FILENAME,
    PipelineConfig,
    build_argument_parser,
    config_from_arguments,
)
from sc4000.dataset_io import dataset_image_dir, map_keypoints_from_rotated_to_original
from sc4000.step2_outlier_removal_and_pair_selection import load_step2_outputs
from sc4000.step3_absolute_rotation_from_pairs import load_step3_absolute_rotation


class AlikedLightGlueMatcher:
    def __init__(self, config: PipelineConfig):
        from lightglue import ALIKED, LightGlue

        self.device = torch.device(config.device if torch.cuda.is_available() else "cpu")
        self.resize_long_side = config.local_feature_resize_long_side
        self.min_match_score = config.lightglue_min_match_score
        self.extractor = ALIKED(
            max_num_keypoints=config.aliked_max_num_keypoints,
            detection_threshold=config.aliked_detection_threshold,
        ).eval().to(self.device)
        self.matcher = LightGlue(
            features="aliked",
            depth_confidence=config.lightglue_depth_confidence,
            width_confidence=config.lightglue_width_confidence,
        ).eval().to(self.device)

    @torch.no_grad()
    def extract_features_after_rotation(self, image_path: Path, rotation_k: int) -> dict:
        """Features in the rotated frame (for LightGlue) plus keypoints mapped back to the original image."""
        from lightglue.utils import load_image

        original_image = load_image(image_path)  # float [3, H, W] in [0, 1]
        _, original_height, original_width = original_image.shape
        rotated_image = torch.rot90(original_image, rotation_k, dims=(1, 2)).contiguous()
        features = self.extractor.extract(rotated_image.to(self.device), resize=self.resize_long_side)
        keypoints_in_rotated_image = features["keypoints"][0].cpu().numpy()
        return {
            "features_in_rotated_frame": {
                "keypoints": features["keypoints"].cpu(),
                "descriptors": features["descriptors"].cpu(),
                "image_size": features["image_size"].cpu(),
            },
            "keypoints_in_original_image": map_keypoints_from_rotated_to_original(
                keypoints_in_rotated_image, original_height, original_width, rotation_k
            ).astype(np.float32),
        }

    @torch.no_grad()
    def match_feature_pair(self, features_a: dict, features_b: dict) -> np.ndarray:
        """int64 [M, 2] keypoint index pairs with LightGlue score >= min_match_score."""
        to_device = lambda features: {key: value.to(self.device) for key, value in features.items()}
        output = self.matcher({"image0": to_device(features_a), "image1": to_device(features_b)})
        matches = output["matches"][0].cpu().numpy()
        scores = output["scores"][0].cpu().numpy()
        return matches[scores >= self.min_match_score].astype(np.int64)


def run_step4_local_matching(
    config: PipelineConfig, split: str, dataset_name: str, local_matcher: "AlikedLightGlueMatcher | None" = None
) -> None:
    _, selected_pairs = load_step2_outputs(config, dataset_name)
    absolute_rotation_k_by_name = load_step3_absolute_rotation(config, dataset_name)
    output_dir = config.dataset_working_dir(dataset_name)
    keypoints_path = output_dir / STEP4_KEYPOINTS_FILENAME
    matches_path = output_dir / STEP4_MATCHES_FILENAME
    keypoints_path.unlink(missing_ok=True)
    matches_path.unlink(missing_ok=True)

    images_in_pairs = sorted({name for pair in selected_pairs for name in pair})
    if not selected_pairs:
        print(f"[step4] {dataset_name}: no selected pairs, nothing to match")
        h5py.File(keypoints_path, "w").close()
        h5py.File(matches_path, "w").close()
        return

    local_matcher = local_matcher or AlikedLightGlueMatcher(config)
    image_dir = dataset_image_dir(config, split, dataset_name)
    features_by_name = {}
    with h5py.File(keypoints_path, "w") as keypoints_file:
        for image_name in tqdm(images_in_pairs, desc="[step4] ALIKED", dynamic_ncols=True):
            features = local_matcher.extract_features_after_rotation(
                image_dir / image_name, absolute_rotation_k_by_name.get(image_name, 0)
            )
            features_by_name[image_name] = features["features_in_rotated_frame"]
            keypoints_file.create_dataset(image_name, data=features["keypoints_in_original_image"])

    kept_pair_count = 0
    with h5py.File(matches_path, "w") as matches_file:
        for image_a, image_b in tqdm(selected_pairs, desc="[step4] LightGlue", dynamic_ncols=True):
            matches = local_matcher.match_feature_pair(features_by_name[image_a], features_by_name[image_b])
            if len(matches) < config.min_matches_per_pair:
                continue
            matches_file.require_group(image_a).create_dataset(image_b, data=matches)
            kept_pair_count += 1
    print(f"[step4] {dataset_name}: {len(images_in_pairs)} images, {kept_pair_count}/{len(selected_pairs)} pairs "
          f"with >= {config.min_matches_per_pair} matches")


# ---------------------------------------------------------------------------------------- self tests


def selftest_keypoint_back_mapping() -> None:
    random_generator = np.random.default_rng(0)
    original_image = random_generator.integers(0, 1_000_000, size=(7, 11))
    for rotation_k in range(ROTATION_COUNT):
        rotated_image = np.rot90(original_image, rotation_k)
        rotated_y, rotated_x = np.meshgrid(np.arange(rotated_image.shape[0]), np.arange(rotated_image.shape[1]), indexing="ij")
        keypoints_in_rotated_image = np.stack([rotated_x.ravel(), rotated_y.ravel()], axis=1)
        keypoints_in_original_image = map_keypoints_from_rotated_to_original(
            keypoints_in_rotated_image, *original_image.shape, rotation_k
        ).astype(int)
        values_rotated = rotated_image[keypoints_in_rotated_image[:, 1], keypoints_in_rotated_image[:, 0]]
        values_original = original_image[keypoints_in_original_image[:, 1], keypoints_in_original_image[:, 0]]
        assert np.array_equal(values_rotated, values_original), f"back-mapping wrong for rotation_k={rotation_k}"
    print("[step4 selftest] rot90 keypoint back-mapping: OK for k = 0..3")


def selftest_rotated_copy_matching(config: PipelineConfig, image_path: Path, scratch_dir: Path) -> None:
    """X = image, Y = rot90(X, k) saved to disk. Step 3 would give Y absolute_rotation_k = (4 - k) % 4,
    which makes both upright, so LightGlue should match them densely. Y keypoints, back-mapped to Y's
    own pixels and then mapped into X with the known rotation, must land on the matched X keypoints."""
    from PIL import Image

    local_matcher = AlikedLightGlueMatcher(config)
    original_image = np.asarray(Image.open(image_path).convert("RGB"))
    original_height, original_width = original_image.shape[:2]
    features_x = local_matcher.extract_features_after_rotation(image_path, 0)
    scratch_dir.mkdir(parents=True, exist_ok=True)
    for rotation_k in range(1, ROTATION_COUNT):
        rotated_copy_path = scratch_dir / f"selftest_rotated_copy_k{rotation_k}.png"
        Image.fromarray(np.ascontiguousarray(np.rot90(original_image, rotation_k))).save(rotated_copy_path)
        features_y = local_matcher.extract_features_after_rotation(rotated_copy_path, (ROTATION_COUNT - rotation_k) % ROTATION_COUNT)
        matches = local_matcher.match_feature_pair(features_x["features_in_rotated_frame"], features_y["features_in_rotated_frame"])
        keypoints_y_in_x = map_keypoints_from_rotated_to_original(
            features_y["keypoints_in_original_image"][matches[:, 1]], original_height, original_width, rotation_k
        )
        error = np.linalg.norm(features_x["keypoints_in_original_image"][matches[:, 0]] - keypoints_y_in_x, axis=1)
        print(f"[step4 selftest] rotated copy k={rotation_k}: {len(matches)} matches, "
              f"median error {np.median(error):.2f}px")
        assert len(matches) > 100 and np.median(error) < 2.0, "rotated-copy matching failed"
    print("[step4 selftest] rotated-copy matching: OK")


def main() -> None:
    parser = build_argument_parser(__doc__.splitlines()[1], needs_dataset=False)
    parser.add_argument("--dataset")
    parser.add_argument("--selftest", action="store_true")
    parser.add_argument("--selftest-image", type=Path)
    arguments = parser.parse_args()
    config = config_from_arguments(arguments)
    if arguments.selftest or arguments.selftest_image:
        selftest_keypoint_back_mapping()
        if arguments.selftest_image:
            selftest_rotated_copy_matching(config, arguments.selftest_image, Path(config.working_dir) / "step4_selftest")
        return
    run_step4_local_matching(config, arguments.split, arguments.dataset)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/sc4000/src/sc4000/step5_reconstruction_colmap.py
"""STEP 5 - COLMAP reconstruction (pycolmap 0.6.1 on Kaggle).

1. Import the step-4 keypoints and matches into a fresh COLMAP database with imc25-utils
   `h5_to_db.add_keypoints` / `add_matches` (these require the files to be named keypoints.h5 and
   matches.h5, so they are symlinked into a staging dir).
2. Geometric verification of the imported pairs with `pycolmap.match_exhaustive`.
3. `pycolmap.incremental_mapping`; every model becomes one cluster.

Output: step5_registered_poses.json
  {image_name: {"cluster_index": int, "rotation_matrix": [9], "translation_vector": [3]}}
Images that are not registered are absent and later reported as outliers.
"""

import os
import shutil
import sys

import h5py
import numpy as np

from sc4000.config import (
    STEP4_KEYPOINTS_FILENAME,
    STEP4_MATCHES_FILENAME,
    STEP5_COLMAP_DIRNAME,
    STEP5_REGISTERED_POSES_FILENAME,
    PipelineConfig,
    build_argument_parser,
    config_from_arguments,
)
from sc4000.dataset_io import dataset_image_dir, read_json, write_json


def import_step4_features_into_colmap_database(config: PipelineConfig, split: str, dataset_name: str, database_path) -> int:
    """Returns the number of imported image pairs."""
    if config.imc25_utils_dir not in sys.path:
        sys.path.append(config.imc25_utils_dir)
    from database import COLMAPDatabase
    from h5_to_db import add_keypoints, add_matches

    dataset_dir = config.dataset_working_dir(dataset_name)
    staging_dir = dataset_dir / STEP5_COLMAP_DIRNAME / "h5_import_staging"
    staging_dir.mkdir(parents=True, exist_ok=True)
    for staged_name, step4_filename in (("keypoints.h5", STEP4_KEYPOINTS_FILENAME), ("matches.h5", STEP4_MATCHES_FILENAME)):
        staged_path = staging_dir / staged_name
        staged_path.unlink(missing_ok=True)
        try:
            staged_path.symlink_to((dataset_dir / step4_filename).resolve())
        except OSError:
            shutil.copy(dataset_dir / step4_filename, staged_path)

    database = COLMAPDatabase.connect(str(database_path))
    database.create_tables()
    filename_to_image_id = add_keypoints(
        database, str(staging_dir), str(dataset_image_dir(config, split, dataset_name)), "",
        config.colmap_camera_model, False,
    )
    add_matches(database, str(staging_dir), filename_to_image_id)
    database.commit()
    database.close()
    with h5py.File(staging_dir / "matches.h5", "r") as matches_file:
        return sum(len(matches_file[image_a].keys()) for image_a in matches_file.keys())


def _cam_from_world(image):
    """pycolmap 0.6 exposes cam_from_world as a property, newer versions as a method."""
    return image.cam_from_world() if callable(image.cam_from_world) else image.cam_from_world


def run_step5_reconstruction(config: PipelineConfig, split: str, dataset_name: str) -> None:
    import pycolmap

    dataset_dir = config.dataset_working_dir(dataset_name)
    colmap_dir = dataset_dir / STEP5_COLMAP_DIRNAME
    shutil.rmtree(colmap_dir, ignore_errors=True)
    models_dir = colmap_dir / "models"
    models_dir.mkdir(parents=True, exist_ok=True)
    database_path = colmap_dir / "database.db"
    poses_path = dataset_dir / STEP5_REGISTERED_POSES_FILENAME
    poses_path.unlink(missing_ok=True)  # a failed run must not leave stale poses for step 6

    imported_pair_count = import_step4_features_into_colmap_database(config, split, dataset_name, database_path)
    if imported_pair_count == 0:
        print(f"[step5] {dataset_name}: no matched pairs, skipping reconstruction")
        write_json(poses_path, {})
        return

    pycolmap.match_exhaustive(str(database_path))  # geometric verification of the imported matches
    mapper_options = pycolmap.IncrementalPipelineOptions()
    mapper_options.min_model_size = config.colmap_min_model_size
    mapper_options.max_num_models = config.colmap_max_num_models
    reconstructions = pycolmap.incremental_mapping(
        database_path=str(database_path),
        image_path=str(dataset_image_dir(config, split, dataset_name)),
        output_path=str(models_dir),
        options=mapper_options,
    )

    registered_poses = {}
    for cluster_index, reconstruction in sorted(reconstructions.items()):
        for image in reconstruction.images.values():
            if image.name in registered_poses:  # keep the first (largest) model an image appears in
                continue
            cam_from_world = _cam_from_world(image)
            registered_poses[image.name] = {
                "cluster_index": int(cluster_index),
                "rotation_matrix": np.asarray(cam_from_world.rotation.matrix()).reshape(-1).tolist(),
                "translation_vector": np.asarray(cam_from_world.translation).reshape(-1).tolist(),
            }
    write_json(poses_path, registered_poses)
    image_count = len(os.listdir(dataset_image_dir(config, split, dataset_name)))
    print(f"[step5] {dataset_name}: registered {len(registered_poses)} images in {len(reconstructions)} clusters "
          f"({imported_pair_count} matched pairs, ~{image_count} files in folder)")


def load_step5_registered_poses(config: PipelineConfig, dataset_name: str) -> dict:
    poses_path = config.dataset_working_dir(dataset_name) / STEP5_REGISTERED_POSES_FILENAME
    return read_json(poses_path) if poses_path.exists() else {}


def main() -> None:
    arguments = build_argument_parser(__doc__.splitlines()[1]).parse_args()
    run_step5_reconstruction(config_from_arguments(arguments), arguments.split, arguments.dataset)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/sc4000/src/sc4000/step6_submission_and_scoring.py
"""STEP 6 - write submission.csv from the step-5 poses and, on the train split, score it.

Images without a pose (step-2 outliers or not registered by COLMAP) are written as scene
"outliers" with NaN poses. On the train split the score uses imc25-utils `metric.score` against
train_labels.csv restricted to the processed datasets, since the metric requires every
ground-truth row to be present.
"""

import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sc4000.config import PipelineConfig, build_argument_parser, config_from_arguments
from sc4000.dataset_io import ImagePrediction, labels_csv_path, load_image_predictions_by_dataset
from sc4000.step5_reconstruction_colmap import load_step5_registered_poses


def _array_to_string(values) -> str:
    return ";".join(f"{value:.09f}" for value in values)


def _nan_string(count: int) -> str:
    return ";".join(["nan"] * count)


def fill_predictions_from_step5(config: PipelineConfig, predictions: "list[ImagePrediction]", dataset_name: str) -> None:
    registered_poses = load_step5_registered_poses(config, dataset_name)
    for prediction in predictions:
        pose = registered_poses.get(prediction.filename)
        if pose is None:
            continue
        prediction.cluster_index = pose["cluster_index"]
        prediction.rotation_matrix = np.asarray(pose["rotation_matrix"])
        prediction.translation_vector = np.asarray(pose["translation_vector"])


def write_submission_csv(predictions_by_dataset: "dict[str, list[ImagePrediction]]", split: str, submission_csv_path: Path) -> None:
    rows = []
    for predictions in predictions_by_dataset.values():
        for prediction in predictions:
            row = {
                "dataset": prediction.dataset,
                "scene": "outliers" if prediction.cluster_index is None else f"cluster{prediction.cluster_index}",
                "image": prediction.filename,
                "rotation_matrix": _nan_string(9) if prediction.rotation_matrix is None else _array_to_string(prediction.rotation_matrix),
                "translation_vector": _nan_string(3) if prediction.translation_vector is None else _array_to_string(prediction.translation_vector),
            }
            if split == "test":
                row = {"image_id": prediction.image_id, **row}
            rows.append(row)
    submission_csv_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(submission_csv_path, index=False)
    print(f"[step6] wrote {len(rows)} rows -> {submission_csv_path}")


def score_train_submission(config: PipelineConfig, submission_csv_path: Path, dataset_names: "list[str]") -> float:
    if config.imc25_utils_dir not in sys.path:
        sys.path.append(config.imc25_utils_dir)
    import metric

    ground_truth = pd.read_csv(labels_csv_path(config, "train"))
    ground_truth_subset_path = Path(config.working_dir) / "step6_train_labels_processed_datasets.csv"
    ground_truth[ground_truth.dataset.isin(dataset_names)].to_csv(ground_truth_subset_path, index=False)
    (final_score, _, _), (score_by_dataset, _, _) = metric.score(
        gt_csv=str(ground_truth_subset_path),
        user_csv=str(submission_csv_path),
        thresholds_csv=str(Path(config.competition_data_dir) / "train_thresholds.csv"),
        mask_csv=None,
        inl_cf=0,
        strict_cf=-1,
        verbose=True,
    )
    for dataset_name, dataset_score in score_by_dataset.items():
        print(f"[step6] {dataset_name}: {dataset_score:.2f}")
    print(f"[step6] final score over {len(dataset_names)} datasets: {final_score:.2f}")
    return final_score


def run_step6_submission(config: PipelineConfig, split: str, dataset_names: "list[str]") -> None:
    predictions_by_dataset = load_image_predictions_by_dataset(config, split)
    if split == "train":  # the train submission only covers what was processed, matching the filtered GT
        predictions_by_dataset = {name: predictions_by_dataset[name] for name in dataset_names}
    for dataset_name in dataset_names:
        fill_predictions_from_step5(config, predictions_by_dataset[dataset_name], dataset_name)
    submission_csv_path = Path(config.submission_csv_path)
    write_submission_csv(predictions_by_dataset, split, submission_csv_path)
    if split == "train":
        score_train_submission(config, submission_csv_path, dataset_names)


def main() -> None:
    parser = build_argument_parser(__doc__.splitlines()[1], needs_dataset=False)
    parser.add_argument("--datasets", nargs="+", required=True)
    arguments = parser.parse_args()
    run_step6_submission(config_from_arguments(arguments), arguments.split, arguments.datasets)


if __name__ == "__main__":
    main()


## Run

In [ ]:
IS_TRAIN = False
TRAIN_DATASETS = "ETs stairs imc2023_haiper imc2023_heritage"

In [ ]:
INPUT_FLAGS = (f"--competition-data-dir {COMPETITION_DIR} --imc25-utils-dir {IMC25_UTILS_DIR} "
               f"--dinov3-model-dir {TRAINING_OUTPUT_DIR}/dinov3-vitb16-pretrain-lvd1689m "
               f"--pair-classifier-checkpoint-path {TRAINING_OUTPUT_DIR}/pair_classifier/pair_classifier_head_dinov3.pt")
if IS_TRAIN:
    !python /kaggle/working/sc4000/scripts/run_pipeline.py --split train --datasets {TRAIN_DATASETS} {INPUT_FLAGS} \
        --submission-csv-path /kaggle/working/submission_train.csv
else:
    !python /kaggle/working/sc4000/scripts/run_pipeline.py --split test {INPUT_FLAGS}